# 2. Extracción y Limpieza de Alimentos

Este cuaderno implementa un flujo de trabajo optimizado para transformar la base de datos masiva de **Open Food Facts** (alojada en Hugging Face) en un dataset estructurado listo para el modelado de Machine Learning.

---

##  Fases del Proceso

### 1. Extracción de Datos de Alto Rendimiento
Para manejar un volumen de datos tan grande (más de 4 millones de registros originales), se utiliza el formato **Parquet**.
* **Carga Selectiva**: No descargamos todo el archivo; solo extraemos las columnas críticas: `product_name`, `nutriscore_grade`, `nova_group` y `additives_tags`.
* **Fuente**: Se conecta directamente al repositorio oficial en Hugging Face.

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

In [ ]:
# URL del archivo Parquet en Hugging Face
url = "https://huggingface.co/datasets/openfoodfacts/product-database/resolve/main/food.parquet?download=true"

# Columnas mínimas necesarias
columns = ['product_name', 'nutriscore_grade', 'nova_group', 'additives_tags']

# Carga ultra rápida de solo lo que necesitas
df = pd.read_parquet(url, columns=columns)

### 2. Limpieza y Filtrado de Calidad
El dataset crudo contiene mucho "ruido" (valores nulos o inconsistentes). Se aplican tres filtros de limpieza:
* **Validación de Nutriscore**: Se conservan únicamente los productos con calificaciones reales entre la **'a'** y la **'e'**.
* **Validación de NOVA**: Se filtran los grupos de procesamiento para que estén estrictamente en el rango del **1 al 4**.
* **Normalización de Aditivos**: Se transforman los valores nulos en listas vacías (`[]`).

> **Resultado del filtrado**: Se obtiene un dataset limpio de aproximadamente **836,199 productos**.

In [ ]:
# 1. Aseguramos Nutriscore entre 'a' y 'e'
df['nutriscore_grade'] = df['nutriscore_grade'].str.lower()
valores_validos_nutri = ['a', 'b', 'c', 'd', 'e']
df = df[df['nutriscore_grade'].isin(valores_validos_nutri)].copy()

# 2. Aseguramos NOVA entre 1 y 4
df['nova_group'] = pd.to_numeric(df['nova_group'], errors='coerce')
df = df[df['nova_group'].isin([1, 2, 3, 4])].copy()

# 3. Tratamos los aditivos: convertimos Nulos en listas vacías
# NO filtramos por longitud, así mantenemos los productos con []
df['additives_tags'] = df['additives_tags'].apply(lambda d: d if isinstance(d, (list, np.ndarray)) else [])

print(f"Dataset listo para el modelo. Total productos: {len(df)}")
print(f"Productos sin aditivos (control): {sum(df['additives_tags'].map(len) == 0)}")

Dataset listo para el modelo. Total productos: 836199
Productos sin aditivos (control): 342097


### 3. Ingeniería de Características (One-Hot Encoding)
Para que el modelo pueda procesar los aditivos (que vienen en formato de lista), se realiza una expansión matricial:
* **Vectorización**: Se utiliza `get_dummies()` con un separador pipe (`|`) para crear una columna individual por cada aditivo único encontrado.
* **Matriz Binaria**: El resultado es una matriz donde cada fila representa un producto y cada columna un aditivo, marcando con un **1** la presencia y con **0** la ausencia.


In [ ]:
# Creamos la matriz de aditivos
df_dummies = df['additives_tags'].str.join('|').str.get_dummies()

# Unimos con los scores
# Usamos el índice para asegurar que cada fila se mantenga en su sitio
df_final = pd.concat([
    df[['product_name', 'nutriscore_grade', 'nova_group']].reset_index(drop=True), 
    df_dummies.reset_index(drop=True)
], axis=1)

# Rellenamos posibles NaNs con 0 (por si acaso al concatenar)
df_final = df_final.fillna(0)

### 4. Resumen Estadístico de la Muestra
Tras procesar los datos, el cuaderno genera un análisis de la densidad de la información:
* **Diversidad Química**: Se han identificado **572 tipos de aditivos** diferentes en el mercado.
* **Prevalencia**: El aditivo más frecuente es el **E-330 (Ácido Cítrico)**, presente en más de 159,000 productos.
* **Densidad**: La matriz final tiene una densidad del **0.40%**, lo que indica que es una matriz dispersa (sparse matrix) muy común en análisis de ingredientes complejos.



In [ ]:
# 1. Cargar el dataset que conseguiste
df = pd.read_csv('dataset_800k_aditivos.csv') 

# 2. Identificar columnas de aditivos
cols_aditivos = [c for c in df.columns if c.startswith('en:e')]

# 3. Contar presencias
# Sumamos cada columna: si la suma es > 0, el aditivo existe en tu muestra
conteo = df[cols_aditivos].sum()
aditivos_reales = conteo[conteo > 0].sort_values(ascending=False)

print(f"--- RESUMEN DE LA EXTRACCIÓN ---")
print(f"Total de productos analizados: {len(df)}")
print(f"Total de columnas (aditivos posibles): {len(cols_aditivos)}")
print(f"Aditivos encontrados (con al menos un '1'): {len(aditivos_reales)}")
print(f"Aditivos 'fantasma' (siempre a cero): {len(cols_aditivos) - len(aditivos_reales)}")

print(f"\n--- TOP 10 ADITIVOS MÁS FRECUENTES ---")
print(aditivos_reales.head(10))

# 4. Densidad de la matriz
total_celdas = len(df) * len(cols_aditivos)
celdas_con_uno = aditivos_reales.sum()
print(f"\nDensidad de la matriz: {(celdas_con_uno / total_celdas):.4%}")

--- RESUMEN DE LA EXTRACCIÓN ---
Total de productos analizados: 836199
Total de columnas (aditivos posibles): 572
Aditivos encontrados (con al menos un '1'): 572
Aditivos 'fantasma' (siempre a cero): 0

--- TOP 10 ADITIVOS MÁS FRECUENTES ---
en:e330      159317
en:e322      107868
en:e322i      88715
en:e500       68868
en:e415       56987
en:e471       52318
en:e412       48889
en:e500ii     46862
en:e202       45902
en:e450       36438
dtype: int64

Densidad de la matriz: 0.4019%


### 5. Exportación Final
El resultado se guarda en un archivo `dataset_800k_aditivos.csv`, que sirve como base para el entrenamiento de los modelos de clustering (K-Means) y el posterior análisis toxicológico.

In [ ]:
# Exportamos el dataset final a CSV
df_final.to_csv('../data/dataset_800k_aditivos.csv', index=False)